# Student activity count and mean progress by work mode

This notebook describes how each student's mean learning progress relates to the number of eligible activities completed in playlist and ZPDES. Each plotted point is one student × work mode. This is a descriptive analysis, not a causal model.

## 1. Analysis definition

- Use the raw MIA parquet and retain playlist and ZPDES attempts.
- Retain only the first chronological attempt by each student on each exercise.
- Apply the 30-unique-exercise requirement separately within each student × work mode before filtering activities. All first-attempt exercises in that work mode contribute to this threshold.
- For playlist, each playlist ID is a separate sequence. For ZPDES, each module is a separate sequence.
- Map playlist exercises to UUID-backed activities using `config_mia.json` activity `learning_items`. Local ordinal arrays in `exo_mia.json` are not used as global hierarchy identifiers.
- Within each sequence, group attempts belonging to the same activity. Activity changes do not reset a group: attempts from other activities may occur in between.
- A sequence-specific activity is eligible after at least four unique first-attempt exercises. The same pedagogical activity in two playlist IDs therefore contributes twice.
- Activity progress is `(later-half success rate - first-half success rate) × 100`. For an odd number of exercises, the middle exercise is excluded from the comparison, matching the existing progress notebook.
- Student mean progress is the unweighted mean of eligible activity progress: every activity contributes equally.
- The combined population is used. A mixed-mode student may appear once in each panel if eligible in both modes.

In [19]:
from __future__ import annotations

import gc
import importlib
import sys
from argparse import Namespace
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import scripts.model_work_mode_progress as progress_loader  # noqa: E402
import scripts.work_mode_student_activity_progress as student_progress  # noqa: E402

importlib.reload(progress_loader)
importlib.reload(student_progress)

from scripts.model_work_mode_progress import load_attempts  # noqa: E402
from scripts.work_mode_student_activity_progress import (  # noqa: E402
    build_cumulative_module_progress,
    build_student_activity_progress,
)

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 120)

## 2. Parameters

In [20]:
INPUT_FILE = PROJECT_ROOT / 'data_MIA' / '986-neurips-mia_20260415_100024.parquet'
EXERCISE_CATALOG_JSON = PROJECT_ROOT / 'data_MIA' / 'exo_mia.json'
MODULE_CONFIG_JSON = PROJECT_ROOT / 'data_MIA' / 'config_mia.json'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'student_activity_progress_mia_notebook'

MIN_ACTIVITY_EXERCISES = 4
MIN_STUDENT_EXERCISES_PER_MODE = 30
KEEP_ONLY_SINGLE_MODULE_PLAYLISTS = False
JITTER_WIDTH = 0.22
HISTOGRAM_BIN_WIDTH = 10.0
RANDOM_SEED = 20260701

for required_path in (INPUT_FILE, EXERCISE_CATALOG_JSON, MODULE_CONFIG_JSON):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

args = Namespace(
    input_file=INPUT_FILE,
    input_csv=None,
    data_dir=None,
    exercise_catalog_json=EXERCISE_CATALOG_JSON,
    module_config_json=MODULE_CONFIG_JSON,
    keep_only_single_module_playlists=KEEP_ONLY_SINGLE_MODULE_PLAYLISTS,
)

## 3. Load MIA attempts

In [21]:
attempts = load_attempts(args)
display(
    attempts.groupby('work_mode', observed=True)
    .agg(
        attempt_rows=('exercise_id', 'size'),
        students=('student_id', 'nunique'),
        modules=('module', 'nunique'),
        activities=('activity_id', 'nunique'),
    )
    .reset_index()
)

,work_mode,attempt_rows,students,modules,activities
0,playlist,1633559,15696,25,1151
1,zpdes,3957181,28489,27,1222


## 4. Build activity and student summaries

In [22]:
progress_data = build_student_activity_progress(
    attempts,
    min_activity_exercises=MIN_ACTIVITY_EXERCISES,
    min_student_exercises=MIN_STUDENT_EXERCISES_PER_MODE,
)
activity_progress = progress_data.activity_progress
student_summary = progress_data.student_summary
audit = progress_data.audit
del attempts, progress_data
gc.collect()
display(audit)
display(student_summary.head(10))

,work_mode,first_attempt_rows,eligible_attempt_rows,eligible_activities,students_after_30_exercises,students_with_eligible_activities,students_plotted,retained_student_activities,negative_progress_students,negative_progress_share
0,playlist,1425446,1255320,111313,10072,9819,9819,111313,3700,0.376820
1,zpdes,3240623,2418945,340949,20631,20506,20506,340949,1392,0.067883


,student_id,work_mode,activities,modules,mean_progress,median_activity_progress,unique_exercises,negative_progress
0,00017460-1206-45ee-b1d3-393c72a45220,zpdes,29,3,17.528736,0.0,298,False
1,000aa84a-30dd-4dd9-bcfc-bfbb000be34f,zpdes,2,1,12.500000,12.5,35,False
2,000b4ef0-febe-4830-9c2d-a3945b3a8bd9,zpdes,6,1,50.000000,50.0,67,False
3,000e22e9-87f4-48ec-9c32-5a07fbe1b12c,zpdes,4,1,8.333333,0.0,41,False
4,000f3300-302d-4538-ba28-eccde5180ad2,zpdes,9,1,-5.185185,0.0,64,True
5,000f943a-61b3-4b2d-afc2-fd87c386b71e,playlist,14,3,-3.018707,0.0,197,True
6,0010459d-307b-4579-8384-51c7fcb94e0a,zpdes,9,2,23.148148,25.0,115,False
7,00139b95-073c-43e8-bf3e-5b0a40ea6809,playlist,4,2,3.750000,7.5,66,False
8,0014671f-523c-41a5-8179-202336f20066,playlist,4,1,1.071429,0.0,62,False
9,0014671f-523c-41a5-8179-202336f20066,zpdes,9,1,14.814815,0.0,84,False


## 5. Student-level descriptive summary

In [23]:
student_mode_summary = (
    student_summary.groupby('work_mode', observed=True, as_index=False)
    .agg(
        students=('student_id', 'nunique'),
        median_activities=('activities', 'median'),
        maximum_activities=('activities', 'max'),
        median_unique_exercises=('unique_exercises', 'median'),
        mean_student_progress=('mean_progress', 'mean'),
        median_student_progress=('mean_progress', 'median'),
        negative_progress_share=('negative_progress', 'mean'),
    )
)
display(student_mode_summary)

,work_mode,students,median_activities,maximum_activities,median_unique_exercises,mean_student_progress,median_student_progress,negative_progress_share
0,playlist,9819,8.0,260,87.0,1.768619,1.329365,0.376820
1,zpdes,20506,9.0,503,95.0,18.458672,17.777778,0.067883


## 6. Plot activity count against student mean progress

The x coordinate is jittered only for readability. Hover text reports the exact number of eligible sequence-specific activities. The red area indicates negative mean progress.

In [24]:
mode_order = ('zpdes', 'playlist')
colors = {'zpdes': '#4AAE8A', 'playlist': '#4C8EDA'}
symbols = {'zpdes': 'circle', 'playlist': 'diamond'}
rng = np.random.default_rng(RANDOM_SEED)
missing_modes = set(mode_order).difference(student_summary['work_mode'].unique())
if missing_modes:
    raise ValueError(f'No eligible students for work modes: {sorted(missing_modes)}')

panel_titles = []
for work_mode in mode_order:
    mode_data = student_summary[student_summary['work_mode'].eq(work_mode)]
    negative_share = mode_data['negative_progress'].mean()
    mode_label = work_mode.upper() if work_mode == 'zpdes' else work_mode
    panel_titles.append(
        f'{mode_label} (n={len(mode_data):,} étudiants, '
        f'{negative_share:.0%} en régression)'
    )

figure = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    horizontal_spacing=0.08,
    subplot_titles=panel_titles,
)
for column, work_mode in enumerate(mode_order, start=1):
    mode_data = student_summary[student_summary['work_mode'].eq(work_mode)].copy()
    jitter = rng.uniform(-JITTER_WIDTH, JITTER_WIDTH, size=len(mode_data))
    customdata = mode_data[['activities', 'unique_exercises', 'modules']].to_numpy()
    figure.add_trace(
        go.Scattergl(
            x=mode_data['activities'].to_numpy() + jitter,
            y=mode_data['mean_progress'],
            mode='markers',
            marker={
                'color': colors[work_mode],
                'symbol': symbols[work_mode],
                'size': 6,
                'opacity': 0.48,
                'line': {'width': 0},
            },
            customdata=customdata,
            hovertemplate=(
                'Activités admissibles : %{customdata[0]:,.0f}'
                '<br>Progrès moyen : %{y:.1f} points'
                '<br>Exercices uniques : %{customdata[1]:,.0f}'
                '<br>Modules : %{customdata[2]:,.0f}<extra></extra>'
            ),
            showlegend=False,
        ),
        row=1,
        col=column,
    )
    figure.add_hrect(
        y0=-100,
        y1=0,
        fillcolor='#FDECEC',
        opacity=0.55,
        line_width=0,
        layer='below',
        row=1,
        col=column,
    )
    figure.add_hline(
        y=0,
        line={'color': '#777777', 'width': 1, 'dash': 'dash'},
        row=1,
        col=column,
    )

maximum_activities = int(student_summary['activities'].max())
figure.update_xaxes(
    range=[0.5, maximum_activities + 1],
    title_text='Nombre d’activités admissibles (≥ 4 exercices ; jitter visuel)',
    showgrid=True,
    gridcolor='#E7ECF3',
)
figure.update_yaxes(
    range=[-100, 100],
    title_text='Progrès moyen (points de pourcentage)',
    showgrid=True,
    gridcolor='#E7ECF3',
    row=1,
    col=1,
)
figure.update_layout(
    title='Progrès moyen par étudiant — comparaison ZPDES vs playlist',
    title_x=0.02,
    template='simple_white',
    width=1250,
    height=650,
    margin={'l': 90, 'r': 35, 't': 105, 'b': 80},
    font={'family': 'Arial', 'size': 13, 'color': '#2A3F5F'},
)
ACTIVITY_PROGRESS_SVG_CONFIG = {
    'toImageButtonOptions': {
        'format': 'svg',
        'filename': 'student_activity_count_vs_mean_progress',
        'width': 1250,
        'height': 650,
        'scale': 1,
    },
    'displaylogo': False,
}
figure.show(config=ACTIVITY_PROGRESS_SVG_CONFIG)

## 7. Build the cumulative module-progress index

For each student × work mode × module, eligible activity progress values are averaged so every module has one progress value. Those module means are then summed within student × work mode. The result is a cumulative progress index, not a global success-rate change.

In [25]:
cumulative_data = build_cumulative_module_progress(activity_progress)
module_progress = cumulative_data.module_progress
cumulative_student_progress = cumulative_data.student_progress

cumulative_summary = (
    cumulative_student_progress.groupby('work_mode', observed=True, as_index=False)
    .agg(
        students=('student_id', 'nunique'),
        mean_cumulative_progress=('cumulative_progress', 'mean'),
        median_cumulative_progress=('cumulative_progress', 'median'),
        cumulative_progress_sd=('cumulative_progress', 'std'),
        median_modules=('modules', 'median'),
        maximum_modules=('modules', 'max'),
        negative_progress_share=('negative_cumulative_progress', 'mean'),
    )
)
display(cumulative_summary)
display(module_progress.head(10))

,work_mode,students,mean_cumulative_progress,median_cumulative_progress,cumulative_progress_sd,median_modules,maximum_modules,negative_progress_share
0,playlist,9819,3.101242,1.984127,24.809218,2.0,11,0.387616
1,zpdes,20506,29.986642,25.000000,30.574402,1.0,18,0.080806


,student_id,work_mode,module,module_progress,eligible_activities
0,00017460-1206-45ee-b1d3-393c72a45220,zpdes,Améliorer la compréhension des textes,22.916667,12
1,00017460-1206-45ee-b1d3-393c72a45220,zpdes,Orthographe niveau 1,16.666667,9
2,00017460-1206-45ee-b1d3-393c72a45220,zpdes,Orthographe niveau 2,10.416667,8
3,000aa84a-30dd-4dd9-bcfc-bfbb000be34f,zpdes,Espace et Géométrie,12.500000,2
4,000b4ef0-febe-4830-9c2d-a3945b3a8bd9,zpdes,Réapprentissage du sens des nombres,50.000000,6
5,000e22e9-87f4-48ec-9c32-5a07fbe1b12c,zpdes,Orthographe niveau 1,8.333333,4
6,000f3300-302d-4538-ba28-eccde5180ad2,zpdes,Orthographe niveau 1,-5.185185,9
7,000f943a-61b3-4b2d-afc2-fd87c386b71e,playlist,Comprendre les notions de proportion et de fraction,-4.695767,9
8,000f943a-61b3-4b2d-afc2-fd87c386b71e,playlist,Espace et Géométrie,-50.000000,1
9,000f943a-61b3-4b2d-afc2-fd87c386b71e,playlist,"Organisation et gestion de données, fonctions",12.500000,4


## 8. Compare cumulative-progress distributions

The histograms use identical bins and are normalized to percentages, allowing the differently sized playlist and ZPDES populations to be compared directly. Dashed lines show medians.

In [26]:
histogram_values = cumulative_student_progress['cumulative_progress']
histogram_start = (
    np.floor(histogram_values.min() / HISTOGRAM_BIN_WIDTH) * HISTOGRAM_BIN_WIDTH
)
histogram_end = (
    np.ceil(histogram_values.max() / HISTOGRAM_BIN_WIDTH) * HISTOGRAM_BIN_WIDTH
)
if histogram_start == histogram_end:
    histogram_end = histogram_start + HISTOGRAM_BIN_WIDTH

histogram_figure = go.Figure()
for work_mode in mode_order:
    mode_data = cumulative_student_progress[
        cumulative_student_progress['work_mode'].eq(work_mode)
    ]
    mode_label = work_mode.upper() if work_mode == 'zpdes' else work_mode.capitalize()
    histogram_figure.add_trace(
        go.Histogram(
            x=mode_data['cumulative_progress'],
            name=mode_label,
            histnorm='percent',
            xbins={
                'start': histogram_start,
                'end': histogram_end,
                'size': HISTOGRAM_BIN_WIDTH,
            },
            marker={
                'color': colors[work_mode],
                'line': {'color': 'white', 'width': 0.5},
            },
            opacity=0.58,
            hovertemplate=(
                'Indice cumulé : %{x:.1f}'
                '<br>Élèves dans la classe : %{y:.2f}%<extra>%{fullData.name}</extra>'
            ),
        )
    )
    median_value = float(mode_data['cumulative_progress'].median())
    histogram_figure.add_vline(
        x=median_value,
        line={'color': colors[work_mode], 'width': 2, 'dash': 'dash'},
    )
    histogram_figure.add_annotation(
        x=median_value,
        y=20,
        xref='x',
        yref='y',
        text=f'Médiane {mode_label} : {median_value:.1f}',
        showarrow=False,
        xanchor='left' if work_mode == 'zpdes' else 'right',
        yanchor='middle',
        xshift=8 if work_mode == 'zpdes' else -8,
        font={'color': colors[work_mode], 'size': 13},
        bgcolor='rgba(255,255,255,0.82)',
        borderpad=2,
    )

histogram_figure.update_layout(
    title='Distribution de l’indice cumulé de progression par work mode',
    title_x=0.5,
    xaxis_title=(
        'Indice cumulé de progression '
        '(somme des progrès moyens par module)'
    ),
    yaxis_title='Pourcentage d’élèves',
    barmode='overlay',
    template='simple_white',
    width=1100,
    height=620,
    bargap=0.04,
    legend={'orientation': 'h', 'x': 0.5, 'xanchor': 'center', 'y': 1.08},
    font={'family': 'Arial', 'size': 13, 'color': '#2A3F5F'},
)
histogram_figure.update_xaxes(
    range=[histogram_start, histogram_end],
    showgrid=True,
    gridcolor='#E7ECF3',
)
histogram_figure.update_yaxes(
    showgrid=True,
    gridcolor='#E7ECF3',
)
CUMULATIVE_PROGRESS_SVG_CONFIG = {
    'toImageButtonOptions': {
        'format': 'svg',
        'filename': 'cumulative_progress_distribution',
        'width': 1100,
        'height': 620,
        'scale': 1,
    },
    'displaylogo': False,
}
histogram_figure.show(config=CUMULATIVE_PROGRESS_SVG_CONFIG)

## 9. Interpretation limits

The figures are observational. The work modes may contain different students and different exercises, and students with more activities or modules are a selected group. Mean progress averages activity-level early-to-late changes; it is not a continuous learning slope or a causal effect of work mode. The cumulative index sums percentage-point changes across modules and is therefore not itself a success probability or a percentage-point change in global success.

## 10. Save outputs

In [27]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
activity_progress.to_csv(OUTPUT_DIR / 'activity_progress.csv', index=False)
student_summary.to_csv(OUTPUT_DIR / 'student_activity_progress.csv', index=False)
audit.to_csv(OUTPUT_DIR / 'student_activity_progress_audit.csv', index=False)
student_mode_summary.to_csv(OUTPUT_DIR / 'student_mode_summary.csv', index=False)
module_progress.to_csv(OUTPUT_DIR / 'module_progress.csv', index=False)
cumulative_student_progress.to_csv(
    OUTPUT_DIR / 'cumulative_student_progress.csv',
    index=False,
)
cumulative_summary.to_csv(OUTPUT_DIR / 'cumulative_progress_summary.csv', index=False)
figure.write_html(
    OUTPUT_DIR / 'student_activity_count_vs_mean_progress.html',
    include_plotlyjs='cdn',
)
histogram_figure.write_html(
    OUTPUT_DIR / 'cumulative_progress_histogram.html',
    include_plotlyjs='cdn',
)
print(f'Saved outputs to: {OUTPUT_DIR}')

Saved outputs to: c:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\artifacts\student_activity_progress_mia_notebook
